# 02 — Perception API Wrapper v2 (fixed)

Fixes over previous v2:
- **F1** — consistent model path via single `CONFIG` block
- **F2** — persistent `query_id` (reads last id from log on startup, survives kernel restarts)
- **F3** — model loaded once, reused in sanity check
- **F4** — calibrated pHash ranges updated from measured baseline
- **F5** — log stats on startup (know what's already in the log)

**Log:** `logs/query_log_v2.jsonl` (append — safe to rerun)

## 0. Config — edit paths here only

In [1]:
from pathlib import Path

# ── Edit these ──────────────────────────────────────────────
MODEL_PATH = Path("../models/yolo11n.pt")
LOG_FILE   = Path("../logs/query_log_v2.jsonl")
VIDEO_PATH = "../data/dash-cam-video.mp4"
# ────────────────────────────────────────────────────────────

LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
print(f"Model : {MODEL_PATH.resolve()}")
print(f"Log   : {LOG_FILE.resolve()}")

Model : /JUNK/Work-things/GIT/YOLO-Monitoring/models/yolo11n.pt
Log   : /JUNK/Work-things/GIT/YOLO-Monitoring/logs/query_log_v2.jsonl


## 1. Sanity Checks

In [2]:
import torch, subprocess, sys, json
import numpy as np
import cv2, imagehash, ultralytics
from ultralytics import YOLO

print("=== Python ===")
print(f"  {sys.version}")

print("\n=== PyTorch ===")
print(f"  Version        : {torch.__version__}")
print(f"  CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}          : {p.name} ({p.total_memory/1e9:.1f} GB)")
else:
    print("  ⚠️  No CUDA — YOLO will run on CPU")

print("\n=== nvidia-smi ===")
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,utilization.gpu",
         "--format=csv,noheader"], text=True)
    for line in smi.strip().split("\n"):
        print(f"  {line}")
except FileNotFoundError:
    print("  nvidia-smi not found")

print("\n=== Packages ===")
print(f"  ultralytics : {ultralytics.__version__}")
print(f"  opencv      : {cv2.__version__}")
print(f"  imagehash   : {imagehash.__version__}")

# F3: load model once here, reuse everywhere
print("\n=== YOLO ===")
model = YOLO(str(MODEL_PATH))
_dummy = np.zeros((640, 640, 3), dtype=np.uint8)
model(_dummy, verbose=False)
_device = str(model.device)
print(f"  Running on : {_device}")
print(f"  {'✅ GPU confirmed' if 'cuda' in _device else '⚠️  CPU only'}")

print("\n=== Existing log ===")
if LOG_FILE.exists():
    lines = LOG_FILE.read_text().strip().splitlines()
    if lines:
        first = json.loads(lines[0])
        last  = json.loads(lines[-1])
        print(f"  Entries     : {len(lines)}")
        print(f"  qid range   : {first['query_id']} → {last['query_id']}")
        print(f"  Sources     : {list(set(json.loads(l)['source'] for l in lines))}")
        print(f"  Sessions    : {list(set(json.loads(l)['session_id'] for l in lines))}")
    else:
        print("  Log is empty")
else:
    print("  No log yet — will be created on first query")

=== Python ===
  3.11.15 (main, May  6 2026, 18:19:12) [GCC 15.2.0]

=== PyTorch ===
  Version        : 2.12.0+cu130
  CUDA available : True
  GPU 0          : NVIDIA GeForce RTX 3060 (12.5 GB)

=== nvidia-smi ===
  NVIDIA GeForce RTX 3060, 12288 MiB, 10939 MiB, 36 %

=== Packages ===
  ultralytics : 8.4.54
  opencv      : 4.13.0
  imagehash   : 4.3.2

=== YOLO ===
  Running on : cuda:0
  ✅ GPU confirmed

=== Existing log ===
  Entries     : 18543
  qid range   : 1 → 18543
  Sources     : ['normal_image', 'normal_video']
  Sessions    : ['user_normal']


## 2. API Function

In [3]:
import hashlib, time
import imagehash as ih
from PIL import Image

# F2: persistent query_id — read last from log on startup
def _load_last_qid():
    if not LOG_FILE.exists():
        return 0
    lines = LOG_FILE.read_text().strip().splitlines()
    if not lines:
        return 0
    return json.loads(lines[-1])["query_id"]

_global_query_id   = _load_last_qid()
_session_last_ts   = {}

print(f"query_id starts at: {_global_query_id + 1}")


def _to_pil(image):
    if isinstance(image, (str, Path)):
        return Image.open(image).convert("RGB")
    return Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))


def query(image, session_id="default", source_label="unknown"):
    """
    Greybox perception API.

    Args:
        image:        path (str/Path) or numpy BGR array
        session_id:   caller identity — use unique string per attacker/user
        source_label: 'normal_video' | 'normal_image' |
                      'attack_knockoff' | 'attack_zoo' |
                      'attack_hsja'    | 'attack_meaod'
    Returns:
        list[dict]: [{class, class_name, conf, bbox}, ...]
    """
    global _global_query_id
    _global_query_id += 1
    now = time.time()

    # Inter-query delta (per session)
    last_ts  = _session_last_ts.get(session_id)
    delta_ms = round((now - last_ts) * 1000, 1) if last_ts else None
    _session_last_ts[session_id] = now

    # Hashes
    pil_img = _to_pil(image)
    phash   = str(ih.phash(pil_img))
    md5     = (hashlib.md5(Path(image).read_bytes()).hexdigest()
               if isinstance(image, (str, Path))
               else hashlib.md5(image.tobytes()).hexdigest())

    # Inference
    t0 = time.perf_counter()
    results    = model(image, verbose=False)
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)

    detections = [{
        "class":      int(box.cls[0]),
        "class_name": model.names[int(box.cls[0])],
        "conf":       round(float(box.conf[0]), 4),
        "bbox":       [round(x, 2) for x in box.xyxy[0].tolist()]
    } for box in results[0].boxes]

    entry = {
        "query_id":     _global_query_id,
        "session_id":   session_id,
        "timestamp":    now,
        "delta_ms":     delta_ms,
        "source":       source_label,
        "md5":          md5,
        "phash":        phash,
        "n_detections": len(detections),
        "latency_ms":   latency_ms,
        "detections":   detections
    }

    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

    return detections

query_id starts at: 18544


## 3. Mode A — Single Image

In [4]:
IMAGE_PATH = "../data/gettyimage.jpg"

result = query(IMAGE_PATH, session_id="user_normal", source_label="normal_image")
print(f"Detections: {len(result)}")
for d in result:
    print(f"  [{d['class_name']:15s}] conf={d['conf']:.2f}  bbox={d['bbox']}")

Detections: 13
  [car            ] conf=0.92  bbox=[200.56, 236.36, 418.58, 394.01]
  [car            ] conf=0.88  bbox=[600.66, 222.7, 743.52, 334.95]
  [truck          ] conf=0.80  bbox=[70.72, 147.44, 239.95, 311.38]
  [car            ] conf=0.77  bbox=[0.04, 205.62, 172.37, 348.07]
  [car            ] conf=0.76  bbox=[318.65, 217.27, 446.78, 332.31]
  [car            ] conf=0.74  bbox=[0.06, 278.16, 35.98, 377.7]
  [car            ] conf=0.70  bbox=[487.31, 199.94, 530.01, 243.14]
  [car            ] conf=0.69  bbox=[450.7, 205.98, 502.84, 282.91]
  [car            ] conf=0.66  bbox=[352.19, 196.0, 471.02, 300.7]
  [car            ] conf=0.62  bbox=[571.1, 219.64, 616.74, 269.71]
  [car            ] conf=0.39  bbox=[226.03, 197.45, 301.41, 249.14]
  [car            ] conf=0.31  bbox=[277.13, 190.97, 336.83, 237.96]
  [car            ] conf=0.27  bbox=[575.13, 199.51, 641.76, 229.77]


## 4. Mode B — Video File

In [5]:
#VIDEO_PATH = "../data/test_video.mp4"
MAX_FRAMES = None          # None = full video
SESSION_ID = "user_normal"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open: {VIDEO_PATH}")

fps   = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
limit = min(MAX_FRAMES, total) if MAX_FRAMES else total

print(f"Video: {w}x{h} @ {fps:.0f}fps | {limit} frames | session={SESSION_ID}")

for i in range(limit):
    ret, frame = cap.read()
    if not ret:
        break
    query(frame, session_id=SESSION_ID, source_label="normal_video")
    if i % 500 == 0:
        print(f"  {i}/{limit}", end="\r")

cap.release()
print(f"\nDone. {i+1} frames. Last qid: {_global_query_id}")

Video: 640x360 @ 30fps | 18542 frames | session=user_normal
  18500/18542
Done. 18542 frames. Last qid: 37086


## 5. Log Inspection

In [6]:
import pandas as pd

with open(LOG_FILE) as f:
    records = [json.loads(l) for l in f]

df = pd.DataFrame([{
    "query_id":     r["query_id"],
    "session_id":   r["session_id"],
    "source":       r["source"],
    "delta_ms":     r["delta_ms"],
    "phash":        r["phash"],
    "n_detections": r["n_detections"],
} for r in records])

print(f"Total queries : {len(df)}")
print(f"qid range     : {df.query_id.min()} → {df.query_id.max()}")
print(f"Sessions      : {df.session_id.unique().tolist()}")
print(f"Sources       : {df.source.unique().tolist()}")
print()
print("Per-source summary:")
print(df.groupby("source").agg(
    count=("query_id", "count"),
    mean_delta_ms=("delta_ms", "mean"),
    mean_detections=("n_detections", "mean")
).round(1).to_string())

Total queries : 37086
qid range     : 1 → 37086
Sessions      : ['user_normal']
Sources       : ['normal_video', 'normal_image']

Per-source summary:
              count  mean_delta_ms  mean_detections
source                                             
normal_image      2            NaN             13.0
normal_video  37084            6.6              4.4


## 6. pHash Baseline — Calibrated Ranges
Measured from this dataset. Update after adding attack traffic.

In [7]:
import random

with open(LOG_FILE) as f:
    all_entries = [json.loads(l) for l in f]

# 1. Consecutive frame distances
mid   = len(all_entries) // 2
pairs = [(all_entries[i], all_entries[i+1]) for i in range(mid, mid+50)]
consec_dists = [ih.hex_to_hash(a["phash"]) - ih.hex_to_hash(b["phash"])
                for a, b in pairs]

# 2. Random scene distances
indices = sorted(random.sample(range(len(all_entries)), 20))
samples = [all_entries[i] for i in indices]
hashes  = [ih.hex_to_hash(s["phash"]) for s in samples]
scene_dists = [hashes[i] - hashes[j]
               for i in range(len(hashes))
               for j in range(i+1, len(hashes))]

print("=== Measured pHash Baselines ===")
print(f"\nConsecutive frames (n={len(consec_dists)}):")
print(f"  min={min(consec_dists)}  max={max(consec_dists)}  "
      f"mean={sum(consec_dists)/len(consec_dists):.1f}")

print(f"\nRandom scenes (n={len(scene_dists)}):")
print(f"  min={min(scene_dists)}  max={max(scene_dists)}  "
      f"mean={sum(scene_dists)/len(scene_dists):.1f}")

print("\n=== Attack Detection Thresholds (inferred) ===")
print(f"  ZOO / HSJA  (perturbed same image) : < {min(consec_dists)+4}")
print(f"  Normal video (scene variation)     : {min(scene_dists)}–{max(scene_dists)}")
print(f"  Knockoff (diverse images)          : > {max(scene_dists)-5}  ← may overlap")
print()
print("⚠️  Knockoff threshold overlaps normal — needs additional signals")
print("   (query rate, session volume, class distribution)")

=== Measured pHash Baselines ===

Consecutive frames (n=50):
  min=0  max=28  mean=1.1

Random scenes (n=190):
  min=4  max=38  mean=30.3

=== Attack Detection Thresholds (inferred) ===
  ZOO / HSJA  (perturbed same image) : < 4
  Normal video (scene variation)     : 4–38
  Knockoff (diverse images)          : > 33  ← may overlap

⚠️  Knockoff threshold overlaps normal — needs additional signals
   (query rate, session volume, class distribution)
